# Linea base DNS por servidor

## Objetivo

Crear una linea base de volumen y comportamiento DNS por servidor DNS o controlador de dominio.

## Entradas esperadas

- Ventana de tiempo.
- Granularidad temporal.

## Requisitos

- Acceso al area de trabajo de Microsoft Sentinel.
- Funciones KQL publicadas: `fn_Normalize_Windows_DHCP`, `fn_Normalize_Windows_DNS`, `fn_Correlate_DHCP_DNS`.
- Paquetes Python sugeridos: `msticpy`, `pandas`, `matplotlib`, `plotly`, `networkx` segun el notebook.

## Secciones

1. Consultas por servidor.
2. NXDOMAIN por servidor.
3. Tipos de consulta.
4. Clientes mas activos.


In [ ]:
# Configuracion general - ajustar antes de ejecutar
workspace_id = "REEMPLAZAR_CON_WORKSPACE_ID"
tenant_id = "REEMPLAZAR_CON_TENANT_ID"

# Conexion sugerida con MSTICPy
# import msticpy as mp
# mp.init_notebook(namespace=globals())
# qry_prov = mp.QueryProvider("MSSentinel")
# qry_prov.connect(workspace=workspace_id, tenant_id=tenant_id)


In [ ]:
query_dns_baseline = """
let Lookback = 7d;
let BinSize = 1h;
fn_Normalize_Windows_DNS(Lookback)
| summarize TotalQueries=count(), DistinctClients=dcount(ClientIp), DistinctDomains=dcount(QueryRootDomain), NxdomainCount=countif(toupper(ResponseCode) has_any ("NXDOMAIN", "NAME_ERROR", "3") or RawMessage has_any ("NXDOMAIN", "Name Error")), TxtQueries=countif(toupper(QueryType) == "TXT") by bin(TimeGenerated, BinSize), DeviceName
| extend NxdomainRatio = todouble(NxdomainCount) / todouble(TotalQueries)
| order by TimeGenerated asc
"""
# dns_baseline_df = qry_prov.exec_query(query_dns_baseline)
print(query_dns_baseline)


## Resumen para incidente

Documentar aqui:

- Hallazgos principales.
- Entidades relevantes: IP, hostname, direccion MAC, dominio.
- Evidencia KQL usada.
- Recomendacion: cerrar, monitorear, escalar o contener.
